# Universal Excel RAG & Formula Translator

Fully dynamic system to process **any** Excel workbook:

- **Step 0** — Set your file path and query (only cell you ever edit)
- **Step 1** — Detect sheets, headers, and extract formulas dynamically
- **Step 2** — LLM translates each formula sheet to Pandas code
- **Step 3** — Safe execution with automatic code sanitisation before `exec`
- **Step 4** — LLM chat agent answers your query against the computed data

---
## Step 0 — Configuration
**Only edit this cell.**

In [51]:
FILE_PATH = "Enterprise_Payroll_Realistic.xlsx"
USER_QUERY = "Using plotly, create a box plot showing the distribution of 'Net_Salary_F' grouped by 'Department'."
LLM_MODEL  = "gpt-4.1-mini"
# ────────────────────────────────────────────────────────────────

print(f"File  : {FILE_PATH}")
print(f"Query : {USER_QUERY}")
print(f"Model : {LLM_MODEL}")

File  : Enterprise_Payroll_Realistic.xlsx
Query : Using plotly, create a box plot showing the distribution of 'Net_Salary_F' grouped by 'Department'.
Model : gpt-4.1-mini


---
## Step 1 — Dynamic Schema & Formula Extraction

In [52]:
import os, re, json, warnings, types
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Cannot find '{FILE_PATH}'. Update FILE_PATH in Step 0.")

print(f"✅  Found '{FILE_PATH}'. Loading...")


# ── Helper 1: Robust header-row detection ───────────────────────
def detect_header_row(ws, max_scan=30, min_str_frac=0.50):
    """Return 1-based row index of the first row where ≥50% of
    non-empty cells are strings. Falls back to row 1."""
    for row_idx in range(1, max_scan + 1):
        cells     = list(ws.iter_rows(min_row=row_idx, max_row=row_idx))[0]
        non_empty = [c for c in cells if c.value is not None]
        if not non_empty:
            continue
        str_cells = [c for c in non_empty if isinstance(c.value, str)]
        if len(str_cells) / len(non_empty) >= min_str_frac:
            return row_idx
    return 1


# ── Helper 2: Normalise cell refs A2 / $A$2 → [HeaderName] ─────
def normalise_formula(formula, col_letter_map):
    """Replace local cell refs with [ColumnName]; preserve Sheet!A1 refs."""
    def _replace(match):
        col_letter = re.sub(r'[\$\d]', '', match.group(0))
        name = col_letter_map.get(col_letter)
        return f"[{name}]" if name else match.group(0)
    return re.sub(r"(?<!!)\$?[A-Z]{1,3}\$?\d+", _replace, formula)


# ── Main extraction loop ────────────────────────────────────────
all_sheets          = pd.read_excel(FILE_PATH, sheet_name=None, header=0)
wb                  = openpyxl.load_workbook(FILE_PATH, data_only=False)
global_metadata     = {"sheets": {}}
formula_sheet_names = []

for sheet_name in wb.sheetnames:
    ws             = wb[sheet_name]
    header_row_idx = detect_header_row(ws)
    data_row_idx   = header_row_idx + 1

    header_cells   = list(ws.iter_rows(min_row=header_row_idx, max_row=header_row_idx))[0]
    headers        = [(c.value or "") for c in header_cells]

    col_letter_map    = {get_column_letter(i+1): h for i, h in enumerate(headers) if h}
    vlookup_index_map = {str(i+1): h            for i, h in enumerate(headers) if h}

    # Extract formulas from the first data row
    detected_formulas = {}
    data_rows = list(ws.iter_rows(min_row=data_row_idx, max_row=data_row_idx))
    if data_rows:
        for col_idx, cell in enumerate(data_rows[0], start=1):
            if col_idx > len(headers): break
            col_name = headers[col_idx - 1]
            if not col_name: continue
            if cell.value and str(cell.value).startswith("="):
                detected_formulas[col_name] = normalise_formula(str(cell.value), col_letter_map)

    # Re-read sheet with correct header row if it isn't row 1
    if header_row_idx != 1:
        df_sheet = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=header_row_idx - 1)
        all_sheets[sheet_name] = df_sheet
    else:
        df_sheet = all_sheets[sheet_name]

    global_metadata["sheets"][sheet_name] = {
        "header_row_detected" : header_row_idx,
        "columns"             : {col: str(dt) for col, dt in df_sheet.dtypes.items()},
        "vlookup_column_indices": vlookup_index_map,
        "formulas_detected"   : detected_formulas,
    }

    if detected_formulas:
        formula_sheet_names.append(sheet_name)
        print(f"  📋  '{sheet_name}' | header row={header_row_idx} | {len(detected_formulas)} formula(s)")
    else:
        print(f"  📄  '{sheet_name}' | header row={header_row_idx} | data/reference sheet")

metadata_json = json.dumps(global_metadata, indent=2)
print(f"\n✅  Done. Formula sheets: {formula_sheet_names}")

✅  Found 'Enterprise_Payroll_Realistic.xlsx'. Loading...
  📄  'Employees' | header row=1 | data/reference sheet
  📄  'Benefits' | header row=1 | data/reference sheet
  📄  'Tax_Brackets' | header row=1 | data/reference sheet
  📋  'Payroll_Run' | header row=1 | 6 formula(s)

✅  Done. Formula sheets: ['Payroll_Run']


---
## Step 2 — LLM Formula Translation

In [53]:
import sys
import re
import json
import os
from dotenv import load_dotenv

load_dotenv()
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.utils.model_registry import get_model_object

lm = get_model_object(LLM_MODEL)
print(f"✅  LLM ready: {LLM_MODEL}")

def strip_code_fences(text):
    text = text.strip()
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
    text = re.sub(r"\n?```$", "", text)
    return text.strip()

TRANSLATION_PROMPT = """\
You are an Expert AI Data Engineer. Translate Excel formulas into vectorized Pandas code.

=== GLOBAL WORKBOOK SCHEMA ===
{metadata_json}

=== TARGET SHEET: {sheet_name} ===
Formulas to translate:
{formulas_json}

=== TASK ===
Write Python that computes every formula column for sheet '{sheet_name}'.
Modify `all_sheets['{sheet_name}']` directly.

=== STRICT RULES ===

1. IMPORTS / CONTEXT
   - `pd` and `np` are already available. Do NOT import anything.
   - All sheets are in a dict called `all_sheets`. Access: `all_sheets['Name']`.

2. OUTPUT
   - Raw Python ONLY. No markdown. No ``` fences. No explanations.

3. EXACT-MATCH LOOKUPS  (VLOOKUP last arg = FALSE / 0)
   - Use `pd.merge(left, right[cols], on=key, how='left')`.
   - After merge: assign looked-up col to the `_F` column.

4. APPROXIMATE-MATCH LOOKUPS  (VLOOKUP last arg = TRUE / 1)
   - Use the module-level function `pd.merge_asof(left, right, ...)` — NOT `df.merge_asof(...)`.
   - Before calling pd.merge_asof you MUST:
       a. Cast the left key column to float64.
       b. Cast the right key column to float64.
       c. Sort the left DataFrame by the left key.
       d. Sort the right DataFrame by the right key.
   - For tax-bracket lookups use the lower-bound column (Min_Income) as right_on with direction='backward'.
   - After merge: assign result to the `_F` column, reset_index(drop=True).

5. INDEX/MATCH or XLOOKUP
   - Simple key→value: build a dict and use `.map()`.
   - Multi-column result: use `pd.merge()`.

6. SUMIFS/COUNTIFS/AVERAGEIFS
   - Use `.groupby().agg().map()` or boolean-mask `.sum()`/`.count()`.

7. IF statements
   - Simple: `np.where(cond, true_val, false_val)`
   - Nested:  `np.select([c1, c2, ...], [v1, v2, ...], default=...)`

8. COLUMN ORDER
   - Compute in dependency order: if B needs A, compute A first.

9. AVOID DUPLICATE LABELS (CRITICAL)
   - The target DataFrame already contains placeholder columns (like 'Role_F' or 'Base_Salary_F').
   - Do NOT use `.rename(columns=...)` to rename a merged column to an existing placeholder. It will create duplicate columns and crash Pandas!
   - Instead, explicitly assign the values: `all_sheets['{sheet_name}']['Role_F'] = all_sheets['{sheet_name}']['Role']` 
   - Then drop the temporary merged column immediately: `all_sheets['{sheet_name}'].drop(columns=['Role'], inplace=True)`
   - NEVER drop the primary join keys!
"""

generated_code_map = {}

if not formula_sheet_names:
    print("ℹ️  No formula sheets found — Step 3 will be skipped.")
else:
    for sheet_name in formula_sheet_names:
        formulas_json = json.dumps(
            global_metadata["sheets"][sheet_name]["formulas_detected"], indent=2
        )
        prompt = TRANSLATION_PROMPT.format(
            metadata_json=metadata_json,
            sheet_name=sheet_name,
            formulas_json=formulas_json,
        )
        print(f"\n🤖  Translating '{sheet_name}'...")
        try:
            resp = lm(prompt)
            raw  = resp[0] if isinstance(resp, list) else resp
            code = strip_code_fences(raw)
            generated_code_map[sheet_name] = code
            print(f"--- Code for '{sheet_name}' ---\n{code}\n" + "-"*60)
        except Exception as e:
            print(f"❌  Translation failed for '{sheet_name}': {e}")
            generated_code_map[sheet_name] = ""

print(f"\n✅  Translation done for {len(generated_code_map)} sheet(s).")

✅  LLM ready: gpt-4.1-mini

🤖  Translating 'Payroll_Run'...
--- Code for 'Payroll_Run' ---
all_sheets['Payroll_Run'] = all_sheets['Payroll_Run'].copy()

# Lookup Role_F
all_sheets['Payroll_Run'] = pd.merge(all_sheets['Payroll_Run'], 
                                       all_sheets['Employees'][['Employee_ID', 'Role']], 
                                       on='Employee_ID', 
                                       how='left')
all_sheets['Payroll_Run']['Role_F'] = all_sheets['Payroll_Run']['Role']
all_sheets['Payroll_Run'].drop(columns=['Role'], inplace=True)

# Lookup Base_Salary_F
all_sheets['Payroll_Run'] = pd.merge(all_sheets['Payroll_Run'], 
                                       all_sheets['Employees'][['Employee_ID', 'Base_Salary']], 
                                       on='Employee_ID', 
                                       how='left')
all_sheets['Payroll_Run']['Base_Salary_F'] = all_sheets['Payroll_Run']['Base_Salary']
all_sheets['Payroll_Run'].drop(columns=['Base_Salar

---
## Step 3 — Safe Code Execution

Before `exec`, the generated code is **sanitised**:
- Any `df.merge_asof(...)` method-style call is rewritten to `pd.merge_asof(df, ...)`
- A `safe_merge_asof` wrapper (auto-cast + auto-sort) replaces `pd.merge_asof` inside the sandbox
- A `step3_all_ok` flag blocks Step 4 if any sheet fails

In [54]:
# ── Safe merge_asof: auto-casts + auto-sorts both sides ─────────
def safe_merge_asof(left, right, left_on, right_on, direction="backward", **kwargs):
    left  = left.copy()
    right = right.copy()
    left[left_on]   = left[left_on].astype(float)
    right[right_on] = right[right_on].astype(float)
    left  = left.sort_values(left_on).reset_index(drop=True)
    right = right.sort_values(right_on).reset_index(drop=True)
    return pd.merge_asof(left, right, left_on=left_on, right_on=right_on,
                         direction=direction, **kwargs).reset_index(drop=True)


# ── Code sanitiser ───────────────────────────────────────────────
def sanitise_generated_code(code):
    """
    Fix common LLM mistakes before exec:
    df.merge_asof(right, ...) → pd.merge_asof(df, right, ...)
    BUT leave it alone if it is already pd.merge_asof!
    """
    def repl(m):
        lhs = m.group(1)
        caller = m.group(2)
        if caller == "pd":
            return m.group(0) # It's already correct! Do not modify.
        return f"{lhs} = pd.merge_asof({caller},"

    code = re.sub(
        r"([\w\[\]'\"]+(?:\.[\w]+)*)\s*=\s*([\w\[\]'\"]+(?:\.[\w]+)*)\s*\.merge_asof\s*\(",
        repl,
        code
    )

    return code


# ── Reload fresh DataFrames ──────────────────────────────────────
all_sheets = pd.read_excel(FILE_PATH, sheet_name=None)
for _sn in wb.sheetnames:
    _hdr = global_metadata["sheets"][_sn]["header_row_detected"]
    if _hdr != 1:
        all_sheets[_sn] = pd.read_excel(FILE_PATH, sheet_name=_sn, header=_hdr - 1)

print("🔄  Fresh DataFrames loaded.")


# ── Build sandbox pd namespace (safe_merge_asof replaces merge_asof) ─
sandbox_pd = types.SimpleNamespace(**{k: getattr(pd, k) for k in dir(pd) if not k.startswith("__")})
sandbox_pd.merge_asof = safe_merge_asof   # intercept ALL pd.merge_asof calls


# ── Execution loop ───────────────────────────────────────────────
execution_results = {}
step3_all_ok      = True

for sheet_name in formula_sheet_names:
    raw_code = generated_code_map.get(sheet_name, "")
    if not raw_code:
        print(f"⚠️  Skipping '{sheet_name}' — no code generated.")
        execution_results[sheet_name] = "skipped"
        continue

    # Sanitise before exec
    code = sanitise_generated_code(raw_code)

    if code != raw_code:
        print(f"\n🔧  Code sanitised for '{sheet_name}' (merge_asof rewrite applied).")

    print(f"\n⚙️  Executing code for '{sheet_name}'...")

    sandbox = {
        "all_sheets"     : all_sheets,
        "pd"             : sandbox_pd,   # patched namespace
        "np"             : np,
        "safe_merge_asof": safe_merge_asof,
        "__builtins__"   : {},
    }

    try:
        exec(code, {}, sandbox)          # noqa: S102
        all_sheets = sandbox["all_sheets"]

        df_result = all_sheets[sheet_name]
        print(f"✅  '{sheet_name}' succeeded! Shape: {df_result.shape}")
        print(df_result.head().to_string(index=False))
        execution_results[sheet_name] = "ok"

    except Exception as exc:
        print(f"❌  '{sheet_name}' failed: {exc}")
        print("    Code sent to exec:")
        print(code)
        execution_results[sheet_name] = f"error: {exc}"
        step3_all_ok = False


print("\n=== EXECUTION SUMMARY ===")
for sn, status in execution_results.items():
    icon = "✅" if status == "ok" else ("⚠️" if status == "skipped" else "❌")
    print(f"  {icon}  {sn}: {status}")

print()
if step3_all_ok:
    print("✅  All sheets OK. Ready for Step 4.")
else:
    print("⚠️  Errors above. Fix before Step 4.")

🔄  Fresh DataFrames loaded.

⚙️  Executing code for 'Payroll_Run'...
✅  'Payroll_Run' succeeded! Shape: (250, 9)
Employee_ID  Perf_Score  OT_Hours Role_F  Base_Salary_F  Gross_Salary_F  Health_Deduct_F  Tax_Rate_F  Net_Salary_F
    EMP1229         2.3         5 Junior          45168       47470.728              150        0.22   36877.16784
    EMP1038         3.4         8 Junior          45064       48488.352              150        0.22   37670.91456
    EMP1217         2.8        18 Junior          45866       49244.496              150        0.22   38260.70688
    EMP1032         3.1        14 Junior          45992       49473.504              150        0.22   38439.33312
    EMP1186         3.6         1 Junior          47812       51299.464              150        0.22   39863.58192

=== EXECUTION SUMMARY ===
  ✅  Payroll_Run: ok

✅  All sheets OK. Ready for Step 4.


---
## Step 4 — Dynamic Chat Query
Runs only when Step 3 succeeds for all formula sheets.

In [56]:
import plotly.io as pio
pio.renderers.default = "browser"

In [57]:
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.figure import Figure as MplFigure
from IPython.display import display

if not step3_all_ok:
    print("⛔  Step 4 skipped — Step 3 had errors. Fix them and re-run Step 3 first.")
else:
    # ── 1. Build live schema (NOW WITH FORMULAS INCLUDED) ───────────
    live_schema = {}
    for sn, df in all_sheets.items():
        live_schema[sn] = {
            "columns"      : list(df.columns),
            "dtypes"       : {col: str(dt) for col, dt in df.dtypes.items()},
            "row_count"    : len(df),
            # CRITICAL ADDITION: Pass the formulas so the AI knows the math!
            "formulas"     : global_metadata["sheets"][sn].get("formulas_detected", {}),
            "sample_values": {col: df[col].dropna().head(3).tolist() for col in df.columns},
        }
    live_schema_json = json.dumps(live_schema, indent=2, default=str)

    # ── 2. Upgraded Chat Prompt ───────────────────────────────────────
    CHAT_PROMPT = f"""\
You are an Advanced AI Data Scientist. All Excel formulas have been computed.
The DataFrames are fully populated and ready to query.

=== AVAILABLE DATAFRAMES ===
Access via `all_sheets['SheetName']` or `df_SheetName`.

Live schema (Including the original Excel formulas used to compute them!):
{live_schema_json}

=== USER QUERY ===
{USER_QUERY}

=== RULES ===
1. OUTPUT: Return ONLY raw Python code. No markdown, no fences.
2. CONTEXT: `pd`, `np`, `px` (plotly.express), and `plt` (matplotlib.pyplot) are available. Do NOT import anything.
3. FINAL ANSWER: Store your final result in a variable called exactly `answer`.

=== ADVANCED LOGIC RULES ===
4. "WHAT-IF" SCENARIOS & SIMULATIONS:
   - If the user asks a hypothetical (e.g., "What if Perf_Score was 5.0?"), recalculate downstream dependencies.
   - Look at the `formulas` dictionary to see the math. Create a `.copy()` of the dataframe, apply the hypothetical value, and rewrite the Pandas math to compute new outcomes.

5. APPROXIMATE LOOKUPS:
   - You MUST use `pd.merge_asof()`. Cast both keys to float64, sort BOTH dataframes, and use `direction='backward'`. 
   - ALWAYS merge on the lower-bound column (`Min_Income`), NEVER `Max_Income`.

6. COMPARING SIMULATIONS (CRITICAL TO AVOID CRASHES):
   - You MUST merge the simulated dataframe and the original dataframe together on their primary key BEFORE doing comparisons like `df_merged['New'] > df_merged['Old']`.

7. AGGREGATIONS: Drop NaN before aggregating.

8. VISUALIZATIONS & CHARTS (NEW):
   - If the user asks for a chart, graph, or plot, generate it using `px` (Plotly) or `plt` (Matplotlib).
   - IMPORTANT: Save the figure object directly to the `answer` variable (e.g., `answer = px.bar(...)` or `fig, ax = plt.subplots(); ...; answer = fig`).
   - Do NOT call `answer.show()` or `plt.show()` in your code. Just assign the figure to `answer`.
"""

    print(f"❓  Query: {USER_QUERY}")
    print("\n🤖  Requesting answer code from LLM...")

    query_code = ""
    try:
        resp3      = lm(CHAT_PROMPT)
        raw3       = resp3[0] if isinstance(resp3, list) else resp3
        query_code = strip_code_fences(raw3)
        print(f"\n--- Generated query code ---\n{query_code}\n" + "-"*60)
    except Exception as e:
        print(f"❌  LLM failed: {e}")

    if query_code:
        # Sanitise query code
        query_code = sanitise_generated_code(query_code)

        # Inject plotting libraries into the sandbox
        q_sandbox = {
            "all_sheets": all_sheets, 
            "pd": sandbox_pd,
            "np": np, 
            "px": px, 
            "plt": plt,
            "__builtins__": {}
        }
        
        for sn, df in all_sheets.items():
            q_sandbox["df_" + re.sub(r"\W+", "_", sn)] = df

        try:
            exec(query_code, {}, q_sandbox)   # noqa: S102
            answer = q_sandbox.get("answer", None)

            print("\n" + "="*60)
            print("                  FINAL ANSWER")
            print("="*60)
            print(f"Query: {USER_QUERY}\n")

            def _fmt(v):
                # 1. Handle Plotly Figures
                if isinstance(v, go.Figure):
                    v.show()
                # 2. Handle Matplotlib Figures
                elif isinstance(v, MplFigure):
                    display(v)
                # 3. Handle DataFrames
                elif isinstance(v, pd.DataFrame):
                    display(v) # Use Jupyter's rich table display instead of print
                # 4. Handle Numerics (Currency formatting)
                elif isinstance(v, (int, float, np.integer, np.floating)):
                    fv = float(v)
                    print(f"${fv:,.2f}" if abs(fv) > 1000 else str(fv))
                # 5. Fallback
                else:
                    print(v)

            if answer is None:
                print("⚠️  No `answer` variable was set by the generated code.")
            elif isinstance(answer, dict):
                for k, v in answer.items():
                    print(f"  {k}:"); _fmt(v); print()
            else:
                _fmt(answer)

            print("="*60)

        except Exception as exc:
            print(f"\n❌  Query execution failed: {exc}")
            print("Generated code:"); print(query_code)
    else:
        print("⚠️  No query code to execute.")

❓  Query: Using plotly, create a box plot showing the distribution of 'Net_Salary_F' grouped by 'Department'.

🤖  Requesting answer code from LLM...

--- Generated query code ---
df_merged = pd.merge(df_Payroll_Run[['Employee_ID', 'Net_Salary_F']], df_Employees[['Employee_ID', 'Department']], on='Employee_ID')
answer = px.box(df_merged, x='Department', y='Net_Salary_F', title='Distribution of Net Salary by Department')
------------------------------------------------------------

                  FINAL ANSWER
Query: Using plotly, create a box plot showing the distribution of 'Net_Salary_F' grouped by 'Department'.

